# Allora Forge Builder Kit — SOL/USD 8h Topic 10 Optimized

Notebook này là bản cập nhật của `Allora Forge Builder Kit.ipynb` để chạy thuật toán từ `SOL_USD_8h_Topic10_Optimized.ipynb`.

**Mục tiêu:** dự đoán SOL/USD log-return 8 giờ.

- `TOPIC_ID = 38`: testnet SOL 8h, nên chạy trước.
- `TOPIC_ID = 10`: mainnet SOL 8h, chỉ dùng khi kết quả test/eval đủ tốt.

**Thay đổi chính so với notebook gốc:**

| Hạng mục | Notebook gốc | Bản SOL 8h này |
|---|---:|---:|
| Ticker chính | BTC | SOL |
| Ticker phụ | ít hoặc không có | BTC + ETH làm cross-asset signals |
| Horizon | thường 24h | 8h |
| Feature engineering | cơ bản | RSI, Bollinger %B, realized volatility, BTC/ETH correlation, beta, seasonality |
| Validation | cơ bản | walk-forward `TimeSeriesSplit` |
| Model | baseline | LightGBM conservative + early stopping |
| Export | `predict.pkl` | `predict.pkl` có cùng feature pipeline khi live inference |

## 0 — Cài đặt thư viện

Chạy cell này đầu tiên. Nếu runtime đã có thư viện thì pip sẽ bỏ qua hoặc nâng cấp nhẹ.

In [ ]:
%pip install -q allora-forge-builder-kit lightgbm scikit-learn dill scipy matplotlib pandas numpy
print("Cài đặt xong")

## 1 — Cấu hình

Điền `FORGE_API_KEY` trước khi chạy. Có thể lấy key tại developer.allora.network.

In [ ]:
import os
from pathlib import Path

# ====== CHỈNH Ở ĐÂY ======
FORGE_API_KEY = "your_api_key_here"
TOPIC_ID = 38          # 38=testnet SOL 8h | 10=mainnet SOL 8h
MNEMONIC = ""          # để trống -> WorkerManager tự tạo/lưu ví

TRAIN_FROM_MONTH = "2023-01"  # có thể đổi về "2022-01" nếu muốn thêm dữ liệu
VALIDATION_MONTHS = 4
TEST_MONTHS = 2

# Workflow/data config cho SOL 8h
TICKERS = ["solusd", "btcusd", "ethusd"]
TARGET_TICKER = "solusd"
HOURS_NEEDED = 48
NUMBER_OF_INPUT_CANDLES = 48
TARGET_LENGTH = 8

os.environ["ALLORA_API_KEY"] = FORGE_API_KEY
if MNEMONIC:
    os.environ["ALLORA_MNEMONIC"] = MNEMONIC

print(f"Config OK | Topic={TOPIC_ID} | Tickers={TICKERS} | Target={TARGET_LENGTH}h")

## 2 — Imports

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import dill
import lightgbm as lgb
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

from allora_forge_builder_kit import AlloraMLWorkflow, get_api_key

print("Imports OK")

## 3 — Khởi tạo workflow và tải dữ liệu

Workflow được đổi sang SOL/USD 8h, đồng thời thêm BTC và ETH để tạo cross-asset features.

In [ ]:
workflow = AlloraMLWorkflow(
    data_api_key=get_api_key(),
    tickers=TICKERS,
    hours_needed=HOURS_NEEDED,
    number_of_input_candles=NUMBER_OF_INPUT_CANDLES,
    target_length=TARGET_LENGTH,
)

X_train, y_train, X_val, y_val, X_test, y_test = workflow.get_train_validation_test_data(
    from_month=TRAIN_FROM_MONTH,
    validation_months=VALIDATION_MONTHS,
    test_months=TEST_MONTHS,
)

print(f"Train: {X_train.shape}")
print(f"Val  : {X_val.shape}")
print(f"Test : {X_test.shape}")
print(f"First columns: {list(X_train.columns[:10])}")

## 4 — Feature Engineering cho SOL 8h

Quan trọng: hàm `add_custom_features()` bên dưới sẽ được dùng lại trong `predict()` khi chạy live worker. Điều này giúp tránh train/serve skew.

In [ ]:
def _find_col(df: pd.DataFrame, ticker: str, field: str) -> str:
    """Tìm cột OHLCV theo nhiều naming convention phổ biến của Builder Kit."""
    candidates = [
        f"{ticker}_{field}",
        f"{ticker.upper()}_{field}",
        f"{ticker.lower()}_{field}",
        f"{ticker}_{field.lower()}",
        f"{ticker.upper()}_{field.lower()}",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: chọn cột có ticker và field trong tên
    t = ticker.lower()
    f = field.lower()
    matches = [c for c in df.columns if t in c.lower() and f in c.lower()]
    if matches:
        return matches[0]
    raise KeyError(f"Không tìm thấy cột cho {ticker}_{field}. Columns mẫu: {list(df.columns[:20])}")


def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period, min_periods=period).mean()
    loss = (-delta.clip(upper=0)).rolling(period, min_periods=period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))


def add_custom_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    sol_close = _find_col(df, "solusd", "close")
    btc_close = _find_col(df, "btcusd", "close")
    eth_close = _find_col(df, "ethusd", "close")

    sol = pd.to_numeric(df[sol_close], errors="coerce")
    btc = pd.to_numeric(df[btc_close], errors="coerce")
    eth = pd.to_numeric(df[eth_close], errors="coerce")

    # Momentum: RSI
    df["sol_rsi_6"] = compute_rsi(sol, 6)
    df["sol_rsi_14"] = compute_rsi(sol, 14)
    df["sol_rsi_24"] = compute_rsi(sol, 24)

    # Mean reversion: Bollinger %B
    for w in [12, 20, 36]:
        mid = sol.rolling(w, min_periods=w).mean()
        std = sol.rolling(w, min_periods=w).std()
        df[f"sol_bb_pct_{w}"] = (sol - (mid - 2 * std)) / (4 * std + 1e-9)

    # SOL returns + realized volatility
    sol_lr = np.log(sol / sol.shift(1))
    df["sol_rvol_8"] = sol_lr.rolling(8, min_periods=4).std()
    df["sol_rvol_24"] = sol_lr.rolling(24, min_periods=12).std()
    df["sol_rvol_48"] = sol_lr.rolling(48, min_periods=24).std()

    df["sol_ret_1h"] = sol_lr
    df["sol_ret_4h"] = np.log(sol / sol.shift(4))
    df["sol_ret_8h"] = np.log(sol / sol.shift(8))
    df["sol_ret_24h"] = np.log(sol / sol.shift(24))

    # BTC cross-asset signals
    btc_lr = np.log(btc / btc.shift(1))
    df["btc_ret_1h"] = btc_lr
    df["btc_ret_4h"] = np.log(btc / btc.shift(4))
    df["btc_ret_8h"] = np.log(btc / btc.shift(8))
    df["sol_btc_corr_12"] = sol_lr.rolling(12, min_periods=6).corr(btc_lr)
    df["sol_btc_corr_24"] = sol_lr.rolling(24, min_periods=12).corr(btc_lr)

    cov = sol_lr.rolling(24, min_periods=12).cov(btc_lr)
    bv = btc_lr.rolling(24, min_periods=12).var()
    df["sol_btc_beta_24"] = cov / (bv + 1e-12)

    # ETH cross-asset signals
    eth_lr = np.log(eth / eth.shift(1))
    df["eth_ret_1h"] = eth_lr
    df["sol_eth_corr_24"] = sol_lr.rolling(24, min_periods=12).corr(eth_lr)

    # Relative strength SOL/BTC
    ratio = sol / btc
    df["sol_btc_ratio_ret8"] = np.log(ratio / ratio.shift(8))

    # Time seasonality, nếu index là DatetimeIndex
    if isinstance(df.index, pd.DatetimeIndex):
        df["hour_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df.index.hour / 24)
        df["dow_sin"] = np.sin(2 * np.pi * df.index.dayofweek / 7)
        df["dow_cos"] = np.cos(2 * np.pi * df.index.dayofweek / 7)

    df = df.replace([np.inf, -np.inf], np.nan)
    return df


X_train_fe = add_custom_features(X_train).dropna()
X_val_fe = add_custom_features(X_val).dropna()
X_test_fe = add_custom_features(X_test).dropna()

y_train_fe = y_train.loc[X_train_fe.index]
y_val_fe = y_val.loc[X_val_fe.index]
y_test_fe = y_test.loc[X_test_fe.index]

FEATURE_COLS = list(X_train_fe.columns)

print(f"Raw features từ workflow: {len(X_train.columns)}")
print(f"Custom features thêm vào: {len(FEATURE_COLS) - len(X_train.columns)}")
print(f"Tổng features: {len(FEATURE_COLS)}")
print(f"Sau dropna | Train={X_train_fe.shape} Val={X_val_fe.shape} Test={X_test_fe.shape}")

## 5 — Train LightGBM với walk-forward CV

Các tham số được đặt conservative để giảm overfit trên crypto data.

In [ ]:
LGBM_PARAMS = dict(
    objective="regression",
    metric="rmse",
    num_leaves=15,
    learning_rate=0.02,
    max_depth=5,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=0.2,
    n_estimators=1000,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

print("Walk-forward CV: TimeSeriesSplit(n_splits=5)")
tscv = TimeSeriesSplit(n_splits=5)
cv_rmse = []
X_np = X_train_fe[FEATURE_COLS].values
y_np = y_train_fe.values

for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X_np), start=1):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(
        X_np[tr_idx], y_np[tr_idx],
        eval_set=[(X_np[vl_idx], y_np[vl_idx])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred = m.predict(X_np[vl_idx])
    rmse = np.sqrt(mean_squared_error(y_np[vl_idx], pred))
    cv_rmse.append(rmse)
    print(f"Fold {fold}: RMSE={rmse:.6f} best_iter={m.best_iteration_}")

print(f"\nCV RMSE mean={np.mean(cv_rmse):.6f} std={np.std(cv_rmse):.6f}")

# Final model: train trên train+val để tận dụng dữ liệu trước test
X_full = pd.concat([X_train_fe[FEATURE_COLS], X_val_fe[FEATURE_COLS]])
y_full = pd.concat([y_train_fe, y_val_fe])

model = lgb.LGBMRegressor(**LGBM_PARAMS)
model.fit(
    X_full,
    y_full,
    eval_set=[(X_val_fe[FEATURE_COLS], y_val_fe)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)],
)

print(f"\nFinal model best_iteration={model.best_iteration_}")

## 6 — Evaluation: 7 metrics grade system

Mục tiêu trước khi deploy mainnet: tối thiểu **B+ = 5/7**.

In [ ]:
def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))


def czar_score(y_true, y_pred):
    threshold = np.percentile(np.abs(y_true), 75)
    mask = np.abs(y_true) > threshold
    if mask.sum() == 0:
        return 0.0
    return directional_accuracy(y_true[mask], y_pred[mask])


def wrmse_vs_zero(y_true, y_pred):
    w = np.abs(y_true) + 1e-9
    model_err = np.sqrt(np.average((y_true - y_pred) ** 2, weights=w))
    zero_err = np.sqrt(np.average(y_true ** 2, weights=w))
    return (zero_err - model_err) / (zero_err + 1e-9)


def bootstrap_da_ci(y_true, y_pred, n=1000, ci=0.95):
    rng = np.random.default_rng(42)
    size = len(y_true)
    samples = []
    for _ in range(n):
        idx = rng.integers(0, size, size)
        samples.append(directional_accuracy(y_true[idx], y_pred[idx]))
    alpha = 1 - ci
    return np.percentile(samples, [alpha / 2 * 100, (1 - alpha / 2) * 100])


y_true = y_test_fe.values
y_pred = model.predict(X_test_fe[FEATURE_COLS])

da = directional_accuracy(y_true, y_pred)
r_val, p_val = pearsonr(y_true, y_pred)
wrmse_imp = wrmse_vs_zero(y_true, y_pred)
czar = czar_score(y_true, y_pred)
da_ci = bootstrap_da_ci(y_true, y_pred)

checks = {
    "DA >= 52%": da >= 0.52,
    "Pearson r >= 0.05": r_val >= 0.05,
    "Pearson p < 0.05": p_val < 0.05,
    "WRMSE >= 5%": wrmse_imp >= 0.05,
    "CZAR >= 10%": czar >= 0.10,
    "DA CI lower >= 0.50": da_ci[0] >= 0.50,
    "DA CI upper >= 0.55": da_ci[1] >= 0.55,
}

passed = sum(checks.values())
grades = {7: "A+", 6: "A", 5: "B+", 4: "B", 3: "C+", 2: "C", 1: "D", 0: "F"}
grade = grades.get(passed, "F")

print("=" * 64)
print(f"EVALUATION — {len(y_true)} test samples")
print("=" * 64)
rows = [
    ("Directional Accuracy", f"{da:.1%}", checks["DA >= 52%"]),
    ("Pearson r", f"{r_val:.4f}", checks["Pearson r >= 0.05"]),
    ("Pearson p-value", f"{p_val:.4f}", checks["Pearson p < 0.05"]),
    ("WRMSE improvement", f"{wrmse_imp:.1%}", checks["WRMSE >= 5%"]),
    ("CZAR large moves", f"{czar:.1%}", checks["CZAR >= 10%"]),
    ("DA 95% CI lower", f"{da_ci[0]:.4f}", checks["DA CI lower >= 0.50"]),
    ("DA 95% CI upper", f"{da_ci[1]:.4f}", checks["DA CI upper >= 0.55"]),
]
for name, value, ok in rows:
    print(f"{name:<26} {value:>10} {'OK' if ok else '--'}")
print("=" * 64)
print(f"GRADE: {grade} ({passed}/7)")
print("=" * 64)

if passed < 5:
    print("\nGợi ý cải thiện trước mainnet:")
    print("- Đổi TRAIN_FROM_MONTH về 2022-01 hoặc 2021-01 nếu API/data hỗ trợ")
    print("- Tăng VALIDATION_MONTHS lên 6")
    print("- Thêm features on-chain/funding/open interest nếu có nguồn dữ liệu ổn định")

## 7 — Feature importance

In [ ]:
imp_df = (
    pd.DataFrame({"feature": FEATURE_COLS, "importance": model.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
ax.set_xlabel("Importance")
ax.set_title("Top 20 Feature Importance")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(imp_df.head(10).to_string(index=False))

## 8 — Export `predict.pkl`

`predict()` sẽ fetch live features, chạy đúng `add_custom_features()`, reindex đúng `FEATURE_COLS`, rồi predict.

In [ ]:
def fetch_live_features_safely(workflow):
    """Tương thích với nhiều version của AlloraMLWorkflow."""
    try:
        return workflow.get_live_features()
    except TypeError:
        return workflow.get_live_features(TARGET_TICKER)


def make_predict_fn(workflow, model, feature_cols, fe_fn):
    def predict():
        live_raw = fetch_live_features_safely(workflow)
        live_fe = fe_fn(live_raw).dropna()
        if live_fe.empty:
            print("WARNING: live_features rỗng sau dropna")
            return pd.Series(dtype=float)

        missing = [c for c in feature_cols if c not in live_fe.columns]
        if missing:
            raise ValueError(
                f"Live features thiếu {len(missing)} cột so với training. "
                f"Ví dụ: {missing[:10]}"
            )

        X_live = live_fe.reindex(columns=feature_cols)
        preds = model.predict(X_live)
        return pd.Series(preds, index=X_live.index, name="prediction")

    return predict


predict = make_predict_fn(workflow, model, FEATURE_COLS, add_custom_features)

with open("predict.pkl", "wb") as f:
    dill.dump(predict, f)

print("Đã lưu predict.pkl")

with open("predict.pkl", "rb") as f:
    predict_loaded = dill.load(f)
print("Load lại predict.pkl thành công")

## 9 — Deploy worker

Chỉ chạy sau khi đã tạo `predict.pkl`. Khuyến nghị chạy testnet topic 38 ít nhất 24h trước khi chuyển `TOPIC_ID = 10`.

In [ ]:
from pathlib import Path

try:
    from allora_forge_builder_kit import WorkerManager, WorkerMonitor, AlloraSDKEventFetcher
except ImportError as e:
    raise ImportError(
        "Không import được WorkerManager. Hãy chắc chắn package allora-forge-builder-kit đã được cài đúng."
    ) from e

artifact_path = Path("predict.pkl")
assert artifact_path.exists(), "Chưa có predict.pkl — hãy chạy cell export trước."

wm = WorkerManager()
result = wm.deploy_worker(topic_id=TOPIC_ID, artifact_path=artifact_path)
print("Deploy result:", result)

# Optional monitor, nếu version package hỗ trợ
try:
    wm.attach_monitor(WorkerMonitor(event_fetcher=AlloraSDKEventFetcher()))
except Exception as e:
    print("Không attach được monitor, bỏ qua:", repr(e))

print("Bắt đầu worker. Dừng notebook/kernel để stop process nếu chạy trong Colab.")
wm.start_all()

## 10 — Monitor gợi ý

Nếu chạy local terminal trong repo, có thể dùng:

```bash
python -m allora_forge_builder_kit.workerctl dashboard
python -m allora_forge_builder_kit.web_dashboard
```

Với dashboard web, mở địa chỉ mà terminal in ra, thường là `http://localhost:8787`.